# Yambda AutoFE — запуск в Google Colab (GPU)

1. **Runtime → Change runtime type → T4 GPU.**
2. Runtime → Run all, разрешить доступ к Google Drive. Держать вкладку открытой.

Порядок: проверка GPU → тесты на утечки → **smoke-прогон на 200 пользователях** (~5–10 мин, проверяет, что всё обучается и ничего не падает) → основной прогон → прогон на 1000 пользователей → отчёт.

На GPU обучается GBDT (XGBoost, `device=cuda`); расчёт признаков идёт на CPU.
Память: в стандартном Colab ≈ 12 ГБ RAM → основной прогон на 5000 пользователей; при High-RAM можно `N_USERS = None` (все 10k).
Результаты сохраняются на Google Drive в `yambda-autofe-results` после каждого эксперимента.

In [ ]:
N_USERS = 5000   # None = все 10k пользователей (нужно >= 25 ГБ RAM)

import os, glob, zipfile, subprocess
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/yambda-autofe-results'
os.makedirs(OUT, exist_ok=True)

# Код: git clone (если репозиторий публичный) -> иначе yambda-autofe.zip с Google Drive -> иначе загрузить zip вручную
REPO = 'https://github.com/georgiken/yambda-autofe.git'
os.chdir('/content')  # при повторном запуске cwd может быть удаляемой папкой /content/recsys
subprocess.run('rm -rf /content/recsys', shell=True)
ok = subprocess.run(f'GIT_TERMINAL_PROMPT=0 git clone -q {REPO} /content/recsys', shell=True).returncode == 0
if not ok:
    zips = glob.glob('/content/drive/MyDrive/**/yambda-autofe*.zip', recursive=True)
    if zips:
        z = zips[0]
    else:
        print('Репозиторий приватный: выберите файл yambda-autofe.zip из папки recsys')
        from google.colab import files
        z = '/content/' + next(iter(files.upload()))
    zipfile.ZipFile(z).extractall('/content/recsys')
    print('код взят из', z)
%cd /content/recsys
!pip install -q -r requirements.txt 2>&1 | tail -2
!nproc; free -g

In [ ]:
# 0) проверка GPU: XGBoost должен обучаться на cuda
!nvidia-smi --query-gpu=name,memory.total --format=csv
import numpy as np, xgboost as xgb
X = np.random.rand(20000, 20); y = (X[:, 0] + np.random.rand(20000) > 1).astype(int)
b = xgb.train({'tree_method': 'hist', 'device': 'cuda', 'objective': 'binary:logistic'}, xgb.QuantileDMatrix(X, y), 20)
print('xgboost', xgb.__version__, 'GPU OK:', b.save_config().count('cuda') > 0)

In [ ]:
# 1) тесты на утечки (~15 с)
!python -m pytest -q tests

In [ ]:
# 2) SMOKE: весь пайплайн на 200 реальных пользователях (сначала скачается Yambda-50M ~ несколько минут)
!python scripts/run_experiments.py --smoke --backend xgb --device cuda --out /content/smoke/u200 2>&1 | grep -v Warning
!python scripts/make_report.py /content/smoke > /dev/null && head -40 /content/smoke/report/tables.md

Если smoke прошёл до строки `done` и метрики выглядят разумно (AUC заметно > 0.5), запускаем основной прогон.

In [ ]:
# 3) основной эксперимент (GPU)
main = 'u10000' if N_USERS is None else f'u{N_USERS}'
flag = '' if N_USERS is None else f'--n-users {N_USERS}'
!python scripts/run_experiments.py {flag} --backend xgb --device cuda --out {OUT}/{main} 2>&1 | grep -v Warning

In [ ]:
# 4) масштабируемость: 1000 пользователей (полные истории)
!python scripts/run_experiments.py --n-users 1000 --backend xgb --device cuda --out {OUT}/u1000 2>&1 | grep -v Warning

In [ ]:
# 5) таблицы и графики + архив
!rm -f {OUT}/events_*.parquet
!python scripts/make_report.py {OUT}
!cd {OUT} && zip -qr /content/results.zip . && ls -la /content/results.zip
from google.colab import files; files.download('/content/results.zip')